# 01 — Baseline POP

Este notebook executa o experimento do baseline **POP** utilizando os perfis produzidos em `experiments/01_preprocessing/02_build_profiles_baseline.ipynb`.

O fluxo foi separado da implementação reutilizável em `src/`: geração do ranking, avaliação semântica com SBERT, pareamento global greedy 1-para-1, cálculo das métricas e salvamento dos resultados.

A avaliação preserva o ranking produzido pelo baseline. O SBERT é usado apenas para estabelecer correspondências com o ground truth e para as métricas semânticas.

## 1. Dependências

In [ ]:
%pip install -q sentence-transformers pandas

## 2. Configuração do projeto

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd

current = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [current, *current.parents] if (p / "src").exists()), current)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sentence_transformers import SentenceTransformer
from src.evaluation.semantic_matching import (
    SBERT_MODEL_NAME,
    THRESHOLD_SBERT,
    carregar_qrels,
    construir_vocabulario,
    encodar_com_sbert,
    construir_matriz_similaridade,
    matching_greedy_1_to_1,
    salvar_npz,
    salvar_csv_avaliacoes_gerais,
)
from src.evaluation.metrics import (
    THRESHOLD_COBERTURA,
    carregar_perfis_por_documento,
    carregar_campos_semanticos_por_documento,
    avaliar_autor,
    salvar_csv_metricas,
)

print(f"Raiz do projeto: {PROJECT_ROOT}")
from src.baselines.pop import gerar_rankings_pop, preparar_tags_brutas, salvar_tags_brutas

## 3. Caminhos e conjunto experimental

In [ ]:
DATA_DIR = PROJECT_ROOT / "data" / "processed"
BASELINE_DIR = DATA_DIR / "baseline"
GROUND_TRUTH_DIR = DATA_DIR / "ground_truth"

AUTHOR_PROFILES = BASELINE_DIR / "perfis_autores.json"
DOCUMENT_PROFILES = BASELINE_DIR / "perfis_por_documento.json"
FILTERED_DOCUMENTS = DATA_DIR / "filtered_documents.json"
QRELS = GROUND_TRUTH_DIR / "LExR-prof-qrels_filtrado"

# Opcional: arquivo JSON contendo a lista de IDs de autores a avaliar.
# Deixe como None para reproduzir a avaliação no conjunto completo.
AUTHOR_IDS_FILE = None

for name, path in {
    "perfis_autores.json": AUTHOR_PROFILES,
    "perfis_por_documento.json": DOCUMENT_PROFILES,
    "filtered_documents.json": FILTERED_DOCUMENTS,
    "LExR-prof-qrels_filtrado": QRELS,
}.items():
    print(f"{name:35s} -> {'OK' if path.exists() else 'não encontrado'}")

MODEL_NAME = "POP"
RESULT_DIR = PROJECT_ROOT / "results" / "baselines" / "pop"
RAW_RANKING = RESULT_DIR / "tags_brutas_pop.json"
DETAIL_CSV = RESULT_DIR / "avaliacao_detalhada_pop.csv"
METRICS_CSV = RESULT_DIR / "metricas_pop.csv"
SIM_DIR = RESULT_DIR / "sim_matrices"

RESULT_DIR.mkdir(parents=True, exist_ok=True)

## 4. Carregamento dos perfis

In [ ]:
with AUTHOR_PROFILES.open("r", encoding="utf-8") as f:
    perfis = json.load(f)

if AUTHOR_IDS_FILE is not None:
    with Path(AUTHOR_IDS_FILE).open("r", encoding="utf-8") as f:
        selected = json.load(f)
    if isinstance(selected, dict):
        selected = list(selected.keys())
    selected = {str(a).replace("ID_", "") for a in selected}
    perfis = {a: tags for a, tags in perfis.items() if str(a).replace("ID_", "") in selected}

print(f"Autores considerados no experimento: {len(perfis):,}")

## 5. Geração do ranking POP

O POP usa a frequência absoluta do n-grama no perfil do pesquisador. O ranking é ordenado por frequência decrescente, com desempate alfabético. Nenhuma normalização adicional é aplicada.

In [ ]:
rankings = gerar_rankings_pop(perfis)
tags_brutas = preparar_tags_brutas(rankings)
salvar_tags_brutas(tags_brutas, RAW_RANKING)

print(f"Rankings POP gerados: {len(rankings):,}")
print(f"Tags brutas: {RAW_RANKING}")

## 6. Preparação da avaliação semântica

O ground truth é carregado dos qrels filtrados. Para Coverage, são usados tanto os n-gramas por documento quanto os campos textuais naturais das publicações. O vocabulário é codificado uma única vez com `paraphrase-multilingual-mpnet-base-v2`.

In [ ]:
gt_norm, gt_original = carregar_qrels(QRELS)
docs_ngrams = carregar_perfis_por_documento(DOCUMENT_PROFILES)
docs_campos = carregar_campos_semanticos_por_documento(
    FILTERED_DOCUMENTS,
    autores_alvo=set(rankings.keys()),
)

# Mantém somente autores efetivamente avaliados.
gt_norm = {a: gt_norm[a] for a in rankings if a in gt_norm}
gt_original = {a: gt_original.get(a, {}) for a in rankings if a in gt_norm}

vocabulario = construir_vocabulario(
    rankings=rankings,
    gt_norm=gt_norm,
    perfis_doc_semantico=docs_campos,
    top_k_pred=20,
)

print(f"Autores com ground truth: {len(gt_norm):,}")
print(f"Vocabulário semântico único: {len(vocabulario):,}")

## 7. Embeddings SBERT

In [ ]:
sbert = SentenceTransformer(SBERT_MODEL_NAME)
cache_emb = encodar_com_sbert(sbert, vocabulario)
print(f"Embeddings calculados: {len(cache_emb):,}")

## 8. Pareamento e cálculo das métricas

O pareamento é global greedy 1-para-1, com limiar 0,75. As matrizes de similaridade são salvas por autor para permitir auditoria e reaproveitamento posterior.

In [ ]:
dados_por_autor = {}
matching_por_autor = {}
metricas_por_autor = {}

SIM_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

for autor, ranking in rankings.items():
    if autor not in gt_norm:
        continue

    dados = construir_matriz_similaridade(
        autor=autor,
        ranking=ranking,
        gt_norm_autor=gt_norm[autor],
        cache_emb=cache_emb,
        top_k=20,
    )
    if dados is None:
        continue

    matched_weights, matched_idx, matched_sims = matching_greedy_1_to_1(
        dados["sim"],
        dados["gold_weights"],
        theta=THRESHOLD_SBERT,
    )

    dados_por_autor[autor] = dados
    matching_por_autor[autor] = {
        "matched_idx": matched_idx,
        "matched_weights": matched_weights,
        "matched_sims": matched_sims,
    }

    salvar_npz(
        dados,
        matched_idx=matched_idx,
        matched_weights=matched_weights,
        matched_sims=matched_sims,
        pasta_saida=SIM_DIR,
    )

    metricas_por_autor[autor] = avaliar_autor(
        dados_autor=dados,
        matched_weights=matched_weights,
        docs_ngrams_autor=docs_ngrams.get(autor, {}),
        docs_campos_autor=docs_campos.get(autor, {}),
        cache_emb=cache_emb,
        theta_cov=THRESHOLD_COBERTURA,
    )

print(f"Autores avaliados: {len(metricas_por_autor):,}")

## 9. Salvamento dos resultados

In [ ]:
salvar_csv_avaliacoes_gerais(
    dados_por_autor=dados_por_autor,
    matching_por_autor=matching_por_autor,
    gt_original=gt_original,
    caminho=DETAIL_CSV,
    modelo=MODEL_NAME,
    rank_max=20,
)

salvar_csv_metricas(
    metricas_por_autor=metricas_por_autor,
    caminho=METRICS_CSV,
    modelo=MODEL_NAME,
)

print(f"Avaliação detalhada: {DETAIL_CSV}")
print(f"Métricas por autor: {METRICS_CSV}")
print(f"Matrizes de similaridade: {SIM_DIR}")

## 10. Resumo agregado

In [ ]:
df_metricas = pd.read_csv(METRICS_CSV, sep=";")
metric_cols = [
    "nDCG@10", "nDCG@20",
    "Precision@5", "Precision@10", "Precision@20",
    "Recall@5", "Recall@10", "Recall@20",
    "MAP@10", "MAP@20",
    "Coverage@10", "Coverage@20",
    "Diversity@10", "Diversity@20",
    "Match_Valido@10", "Match_Valido@20",
]

resumo = df_metricas[metric_cols].mean(numeric_only=True).to_frame("Média")
display(resumo)


## 11. Saídas do experimento

O notebook produz:

- `tags_brutas_pop.json` com o ranking lexical completo;
- `avaliacao_detalhada_pop.csv` com as correspondências por autor e posição;
- `metricas_pop.csv` com as métricas por autor;
- `sim_matrices/` com as matrizes e pareamentos individuais em `.npz`.

Para executar sobre um subconjunto fixo de autores, configure `AUTHOR_IDS_FILE`. Mantendo-o como `None`, o notebook utiliza todos os autores presentes em `perfis_autores.json`.